In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
import os

In [ ]:
# Caminhos
train_path = "/mnt/data/jupyter-env/donaldDuck/src/donald_synthetic_data/splits/train.jsonl"
val_path   = "/mnt/data/jupyter-env/donaldDuck/src/donald_synthetic_data/splits/val.jsonl"

In [ ]:
# Modelo base
MODEL_NAME = "google/gemma-3-4b-it"

In [ ]:
#carga do modelo base e tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.getenv("HF_TOKEN"))

In [ ]:
#carregando modelo base e tokenizer
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=os.getenv("HF_TOKEN"),
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

In [ ]:
# Configuração LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj", "gate_proj",
        "up_proj", "down_proj"
    ]
)

In [ ]:
#Carga e configuração do LoRA
model = get_peft_model(model, lora_config)
model

In [ ]:
# Dataset
data_files = {"train": train_path, "validation": val_path}
dataset = load_dataset("json", data_files=data_files)
dataset

In [ ]:
row_index = 10
print(f'id: {dataset["train"][row_index]["id"]}\n')
print(f'user: {dataset["train"][row_index]["user"]}\n')
print(f'assistant: {dataset["train"][row_index]["assistant"]}\n')
print(f'style_tags: {dataset["train"][row_index]["style_tags"]}\n')
print(f'constraints: {dataset["train"][row_index]["constraints"]}')

In [ ]:
def format_example(example):
    return f"Usuário: {example['user']}\nDonald: {example['assistant']}"

def format_prompt_only(example):
    # Usado para encontrar o índice do corte do loss mask.
    return f"Usuário: {example['user']}\nDonald:"

def tokenize_fn_corrected(example):
    # 1. Obtenha a string completa (Prompt + Resposta)
    full_text = format_example(example)
    
    # 2. Obtenha o prompt (o que o modelo *verá* para começar a responder)
    prompt_text = format_prompt_only(example)
    
    # 3. Tokenize a string completa
    tokenized_full = tokenizer(full_text, truncation=True, padding="max_length", max_length=512)
    
    # 4. Tokenize Apenas o Prompt
    # Adicionamos 'add_special_tokens=False' para evitar que tokens extras (como <bos>) sejam adicionados 
    # e causem um desalinhamento. O tokenizador já adicionará o <bos> no 'full_text' se for o caso.
    tokenized_prompt = tokenizer(prompt_text, truncation=False, add_special_tokens=False)["input_ids"]

    # 5. Crie os rótulos
    labels = tokenized_full["input_ids"].copy()
    
    # 6. Aplique a Máscara (Loss Masking)
    # Defina os tokens do prompt como -100 para que o loss não seja calculado neles.
    # Isso treina o modelo a gerar *apenas* o restante (a resposta do Donald).
    prompt_len = len(tokenized_prompt)
    labels[:prompt_len] = [-100] * prompt_len
    
    # Se você usar padding="max_length", certifique-se de que os tokens de padding também sejam -100
    # Isso geralmente é manipulado automaticamente, mas é uma boa prática.
    labels = [(-100 if token == tokenizer.pad_token_id else token) for token in labels]
    
    tokenized_full["labels"] = labels
    return tokenized_full

tokenized = dataset.map(tokenize_fn_corrected, batched=False)
tokenized

In [ ]:
row_index = 10
print(f'id: {tokenized["train"][row_index]["id"]}\n')
print(f'user: {tokenized["train"][row_index]["user"]}\n')
print(f'assistant: {tokenized["train"][row_index]["assistant"]}\n')
print(f'input_ids: {tokenized["train"][row_index]["input_ids"]}\n')
print(f'attention_mask: {tokenized["train"][row_index]["attention_mask"]}')
print(f'labels: {tokenized["train"][row_index]["labels"]}')

In [ ]:
# Argumentos de treino
args = TrainingArguments(
    output_dir="../donald_lora_out",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    learning_rate=2e-5,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    bf16=True,
    report_to="none",
    optim="paged_adamw_8bit"
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
)

In [ ]:
#treinar
trainer.train()

In [ ]:
#Salvar adapter LoRA
model.save_pretrained("/mnt/data/jupyter-env/donaldDuck/donald_lora_out")
tokenizer.save_pretrained("/mnt/data/jupyter-env/donaldDuck/donald_lora_out")

# Test

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [ ]:
base = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it", torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, "/mnt/data/jupyter-env/donaldDuck/donald_lora_out")
tokenizer = AutoTokenizer.from_pretrained("/mnt/data/jupyter-env/donaldDuck/donald_lora_out")

In [ ]:
user_query = "como foi seu dia?"
prompt = format_prompt_only({"user": user_query})
prompt

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(
    **inputs, 
    max_new_tokens=60,
    do_sample=True,             # Adicione isso para variar a resposta
    temperature=0.7,            # Ajuste a criatividade
    top_k=50,                   # Limite o vocabulário
    eos_token_id=tokenizer.eos_token_id, # Importante!
    pad_token_id=tokenizer.eos_token_id # O Gemma usa o EOS como PAD
)

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

# Pós-processamento para remover o prompt
response_only = generated_text.replace(prompt, "").strip()

print("\n--- Resposta Gerada ---")
print(response_only)